In [32]:
!pip -q install brian2 cirq qsimcirq transformers accelerate datasets sentencepiece \
    openai-whisper ffmpeg-python torchaudio soundfile \
    open-clip-torch timm opencv-python decord

# System dependency: ffmpeg
import shutil, os, sys, subprocess
if shutil.which("ffmpeg") is None:
    print("ffmpeg not found. Install it on your OS:")
    print("  Ubuntu: sudo apt-get update && sudo apt-get install -y ffmpeg")
    print("  macOS:  brew install ffmpeg")
    print("  Win:    choco install ffmpeg")


In [33]:
import zipfile, os, json, glob

zip_path = "/content/data/synthdata.zip"  # Corrected path
out_dir = "./data"

os.makedirs(out_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(out_dir)

root = os.path.join(out_dir, "synthdata", "synth")
print("Root:", root)
print("Files:", sorted(os.listdir(root))[:20])

manifest = os.path.join(root, "manifest.jsonl")
print("Manifest exists:", os.path.exists(manifest))
print("Example line:", open(manifest, "r", encoding="utf-8").readline()[:200])

Root: ./data/synthdata/synth
Files: ['E_aud.npy', 'E_img.npy', 'E_text.npy', 'E_vid.npy', 'audio', 'haptics', 'images', 'manifest.jsonl', 'text', 'video']
Manifest exists: True
Example line: {"i": 0, "timestamp": "2025-08-17T09:56:43.222236", "script_name": "model", "script_id": 4, "cycle": 1, "text": "[model] cycle=1 mood=drifting. classical(H=0.714, M=0.733, S=0.709) quantum(H=0.376, M=


In [34]:
import torch
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
txt_name = "sentence-transformers/all-MiniLM-L6-v2"

txt_tok = AutoTokenizer.from_pretrained(txt_name)
txt_model = AutoModel.from_pretrained(txt_name).to(device).eval()

@torch.no_grad()
def embed_text(texts):
    batch = txt_tok(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    out = txt_model(**batch).last_hidden_state  # (B,T,H)
    mask = batch["attention_mask"].unsqueeze(-1)
    pooled = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return torch.nn.functional.normalize(pooled, dim=-1)


In [35]:
import open_clip
from PIL import Image
import torchvision.transforms as T

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
clip_model = clip_model.to(device).eval()

@torch.no_grad()
def embed_image(paths):
    ims = [clip_preprocess(Image.open(p).convert("RGB")) for p in paths]
    x = torch.stack(ims).to(device)
    z = clip_model.encode_image(x)
    return torch.nn.functional.normalize(z, dim=-1)


In [36]:
import whisper
import numpy as np

wh_model = whisper.load_model("base", device=device)

@torch.no_grad()
def embed_audio(paths):
    embs = []
    for p in paths:
        audio = whisper.load_audio(p)
        audio = whisper.pad_or_trim(audio)
        mel = whisper.log_mel_spectrogram(audio).to(device)
        enc = wh_model.encoder(mel.unsqueeze(0))  # (1, frames, dim)
        emb = enc.mean(dim=1)                     # (1, dim)
        embs.append(emb.squeeze(0))
    embs = torch.stack(embs)
    return torch.nn.functional.normalize(embs, dim=-1)


In [37]:
import decord
decord.bridge.set_bridge("torch")

@torch.no_grad()
def embed_video(paths, num_frames=8):
    all_embs = []
    for p in paths:
        vr = decord.VideoReader(p)
        n = len(vr)
        idx = torch.linspace(0, max(n-1,0), steps=min(num_frames, n)).long()
        frames = vr.get_batch(idx).permute(0,3,1,2)  # (F,C,H,W)
        # preprocess like CLIP
        frames_pil = [T.ToPILImage()(f.cpu()) for f in frames]
        frames_t = torch.stack([clip_preprocess(im) for im in frames_pil]).to(device)
        zf = clip_model.encode_image(frames_t)  # (F,D)
        zv = zf.mean(dim=0, keepdim=True)       # (1,D)
        all_embs.append(zv.squeeze(0))
    all_embs = torch.stack(all_embs)
    return torch.nn.functional.normalize(all_embs, dim=-1)


In [38]:
import numpy as np
from brian2 import *

def brian2_rate_features(x_np, sim_ms=50.0):
    """
    x_np: (B, D) numpy float32 in [-1,1] or [0,1]
    returns: (B, D) spike rates
    """
    B, D = x_np.shape
    # normalize to [0,1] current
    x = x_np
    x = (x - x.min()) / (x.max() - x.min() + 1e-8)

    defaultclock.dt = 0.1*ms
    tau = 10*ms

    eqs = '''
    dv/dt = (-v + I)/tau : 1
    I : 1
    '''

    rates_out = []
    for b in range(B):
        G = NeuronGroup(D, eqs, threshold='v>1', reset='v=0', method='euler')
        G.I = x[b]
        M = SpikeMonitor(G)
        run(sim_ms*ms)
        # spikes per neuron / second
        rate = np.array(M.count) / (sim_ms/1000.0)
        rates_out.append(rate.astype(np.float32))
        # reset Brian2 state between samples
        brian2_device.reinit()
        brian2_device.activate()
    return np.stack(rates_out, axis=0)

In [39]:
import cirq
import qsimcirq
import numpy as np

def quantum_features(x_np, n_qubits=8, reps=2):
    """
    x_np: (B, D) -> use first n_qubits features as angles
    returns: (B, 2*n_qubits) expectation features
    """
    B, D = x_np.shape
    qubits = cirq.LineQubit.range(n_qubits)

    simulator = qsimcirq.QSimSimulator()

    feats = []
    for b in range(B):
        angles = x_np[b, :n_qubits]
        circuit = cirq.Circuit()
        # simple encoding
        for i, q in enumerate(qubits):
            circuit.append(cirq.ry(float(angles[i]))(q))
            circuit.append(cirq.rz(float(angles[i]))(q))

        # entangle
        for _ in range(reps):
            for i in range(n_qubits-1):
                circuit.append(cirq.CNOT(qubits[i], qubits[i+1]))

        # measure expectations via statevector
        result = simulator.simulate(circuit)
        state = result.final_state_vector

        # crude “features”: probabilities of |0> on each qubit (from statevector marginals)
        # For small n_qubits this is OK.
        probs = np.abs(state)**2
        # marginal P(q=0)
        q0 = []
        for qi in range(n_qubits):
            mask = 1 << (n_qubits-1-qi)
            p0 = probs[[i for i in range(len(probs)) if (i & mask)==0]].sum()
            q0.append(p0)
        q0 = np.array(q0, dtype=np.float32)
        feats.append(np.concatenate([q0, 1.0-q0], axis=0))
    return np.stack(feats, axis=0)


In [40]:
import json, os

def load_manifest(root):
    items = []
    with open(os.path.join(root, "manifest.jsonl"), "r", encoding="utf-8") as f:
        for line in f:
            j = json.loads(line)
            # Fix paths if they start with /content/...
            def fix(p):
                if p is None: return None
                p = p.replace("/content/synth/", "")
                return os.path.join(root, p) if not os.path.isabs(p) else p
            items.append({
                "text": j.get("text",""),
                "img": fix(j.get("img_path")),
                "aud": fix(j.get("aud_path")),
                "vid": fix(j.get("vid_path")),
            })
    return items

items = load_manifest(root)
items[0]


{'text': '[model] cycle=1 mood=drifting. classical(H=0.714, M=0.733, S=0.709) quantum(H=0.376, M=0.328, S=0.240).',
 'img': None,
 'aud': None,
 'vid': None}

In [41]:
import torch.nn as nn
import torch.nn.functional as F
import random

class Projector(nn.Module):
    def __init__(self, in_dim, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.Linear(512, out_dim),
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

def info_nce(a, b, temp=0.07):
    logits = (a @ b.T) / temp
    labels = torch.arange(a.size(0), device=a.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

# Filter items to only include entries where all modalities are present
# This is necessary because the original manifest contains entries with missing image, audio, or video paths
filtered_items = [
    item for item in items
    if item["img"] is not None and item["aud"] is not None and item["vid"] is not None and item["text"] is not None
]

if not filtered_items:
    print("Warning: No items found with all modalities (text, image, audio, video). Skipping embedding and training.")
else:
    # Extract modalities from filtered items
    texts = [it["text"] for it in filtered_items]
    imgs  = [it["img"]  for it in filtered_items]
    auds  = [it["aud"]  for it in filtered_items]
    vids  = [it["vid"]  for it in filtered_items]

    E_t = embed_text(texts)
    E_i = embed_image(imgs)
    E_a = embed_audio(auds)
    E_v = embed_video(vids)

    proj_t = Projector(E_t.shape[1]).to(device)
    proj_i = Projector(E_i.shape[1]).to(device)
    proj_a = Projector(E_a.shape[1]).to(device)
    proj_v = Projector(E_v.shape[1]).to(device)

    opt = torch.optim.AdamW(
        list(proj_t.parameters())+list(proj_i.parameters())+list(proj_a.parameters())+list(proj_v.parameters()),
        lr=2e-3, weight_decay=1e-4
    )

    N = E_t.size(0)

    for epoch in range(1, 301):
        idx = torch.randperm(N, device=device)
        # full-batch is fine for N=49
        zt = proj_t(E_t[idx])
        zi = proj_i(E_i[idx])
        za = proj_a(E_a[idx])
        zv = proj_v(E_v[idx])

        loss = (info_nce(zt, zi)+info_nce(zt, za)+info_nce(zt, zv)+
                info_nce(zi, za)+info_nce(zi, zv)+
                info_nce(za, zv))

        opt.zero_grad()
        loss.backward()
        opt.step()

        if epoch % 50 == 0 or epoch == 1:
            print(f"epoch {epoch:4d}  loss {loss.item():.4f}")

In [42]:
!pip -q install brian2 cirq qsimcirq transformers accelerate sentencepiece \
    openai-whisper ffmpeg-python torchaudio soundfile \
    open-clip-torch opencv-python decord pillow

import shutil
if shutil.which("ffmpeg") is None:
    print("ffmpeg not found on OS. Install it:")
    print("Ubuntu: sudo apt-get update && sudo apt-get install -y ffmpeg")
    print("macOS:  brew install ffmpeg")
    print("Win:    choco install ffmpeg")


In [43]:
import os, zipfile, json, numpy as np, pathlib, shutil

ZIP_PATH = "/content/data/synthdata.zip"  # Corrected path
OUT_DIR  = "./data"

# The following lines caused the FileNotFoundError by deleting the zip file before it could be read.
# if os.path.exists(OUT_DIR):
#     shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(OUT_DIR)

ROOT = os.path.join(OUT_DIR, "synthdata", "synth")
manifest_path = os.path.join(ROOT, "manifest.jsonl")

print("ROOT:", ROOT)
print("ROOT contents:", sorted(os.listdir(ROOT)))
print("Manifest:", manifest_path, "exists:", os.path.exists(manifest_path))

items=[]
with open(manifest_path, "r", encoding="utf-8") as f:
    for line in f:
        j=json.loads(line)
        items.append(j)

print("N items:", len(items))
print("Example:", items[0])

ROOT: ./data/synthdata/synth
ROOT contents: ['E_aud.npy', 'E_img.npy', 'E_text.npy', 'E_vid.npy', 'audio', 'haptics', 'images', 'manifest.jsonl', 'text', 'video']
Manifest: ./data/synthdata/synth/manifest.jsonl exists: True
N items: 49
Example: {'i': 0, 'timestamp': '2025-08-17T09:56:43.222236', 'script_name': 'model', 'script_id': 4, 'cycle': 1, 'text': '[model] cycle=1 mood=drifting. classical(H=0.714, M=0.733, S=0.709) quantum(H=0.376, M=0.328, S=0.240).', 'image_path': '/content/synth/images/model_c000001_0000000.png', 'audio_path': '/content/synth/audio/model_c000001_0000000.wav', 'video_path': '/content/synth/video/model_c000001_0000000.mp4'}


In [44]:
def fix_path(p: str) -> str:
    # manifest has /content/synth/images/...
    # your extracted tree is ROOT/images/...
    if p.startswith("/content/synth/"):
        rel = p.replace("/content/synth/", "")
        return os.path.join(ROOT, rel)
    return p

def get_paths(items):
    texts, imgs, auds, vids = [], [], [], []
    for j in items:
        texts.append(j["text"])
        imgs.append(fix_path(j["image_path"]))
        auds.append(fix_path(j["audio_path"]))
        vids.append(fix_path(j["video_path"]))
    return texts, imgs, auds, vids

texts, img_paths, aud_paths, vid_paths = get_paths(items)

print(img_paths[0], os.path.exists(img_paths[0]))
print(aud_paths[0], os.path.exists(aud_paths[0]))
print(vid_paths[0], os.path.exists(vid_paths[0]))


./data/synthdata/synth/images/model_c000001_0000000.png True
./data/synthdata/synth/audio/model_c000001_0000000.wav True
./data/synthdata/synth/video/model_c000001_0000000.mp4 True


In [45]:
import torch
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

TEXT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
txt_tok = AutoTokenizer.from_pretrained(TEXT_MODEL)
txt_model = AutoModel.from_pretrained(TEXT_MODEL).to(device).eval()

@torch.no_grad()
def embed_text(batch_texts):
    batch = txt_tok(batch_texts, padding=True, truncation=True, return_tensors="pt").to(device)
    out = txt_model(**batch).last_hidden_state  # (B,T,H)
    mask = batch["attention_mask"].unsqueeze(-1)
    pooled = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return torch.nn.functional.normalize(pooled, dim=-1)


In [46]:
import open_clip
from PIL import Image

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_model = clip_model.to(device).eval()

@torch.no_grad()
def embed_images(paths):
    ims = [clip_preprocess(Image.open(p).convert("RGB")) for p in paths]
    x = torch.stack(ims).to(device)
    z = clip_model.encode_image(x)
    return torch.nn.functional.normalize(z, dim=-1)


In [47]:
import whisper

wh_model = whisper.load_model("base", device=device)

@torch.no_grad()
def embed_audio(paths):
    embs = []
    for p in paths:
        audio = whisper.load_audio(p)
        audio = whisper.pad_or_trim(audio)
        mel = whisper.log_mel_spectrogram(audio).to(device)
        enc = wh_model.encoder(mel.unsqueeze(0))   # (1, frames, dim)
        emb = enc.mean(dim=1).squeeze(0)          # (dim,)
        embs.append(emb)
    embs = torch.stack(embs)
    return torch.nn.functional.normalize(embs, dim=-1)


In [48]:
import decord
import torchvision.transforms as T
decord.bridge.set_bridge("torch")

@torch.no_grad()
def embed_video(paths, num_frames=8):
    all_embs = []
    for p in paths:
        vr = decord.VideoReader(p)
        n = len(vr)
        idx = torch.linspace(0, max(n-1, 0), steps=min(num_frames, n)).long()
        frames = vr.get_batch(idx).permute(0,3,1,2)  # (F,C,H,W)
        frames_pil = [T.ToPILImage()(f.cpu()) for f in frames]
        frames_t = torch.stack([clip_preprocess(im) for im in frames_pil]).to(device)
        zf = clip_model.encode_image(frames_t)  # (F,D)
        zv = zf.mean(dim=0)
        all_embs.append(zv)
    all_embs = torch.stack(all_embs)
    return torch.nn.functional.normalize(all_embs, dim=-1)


In [49]:
from transformers import pipeline

sent_pipe = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=0 if device=="cuda" else -1)

def sentiment_scores(texts):
    # returns float in [0,1] where 1=positive
    outs = sent_pipe(texts, truncation=True)
    scores = []
    for o in outs:
        if o["label"].upper().startswith("POS"):
            scores.append(float(o["score"]))
        else:
            scores.append(1.0 - float(o["score"]))
    return np.array(scores, dtype=np.float32)

s_scores = sentiment_scores(texts)
s_scores[:5], s_scores.mean()


Device set to use cpu


(array([0.00513488, 0.00370187, 0.00497109, 0.00513488, 0.00445032],
       dtype=float32),
 np.float32(0.25469384))

In [50]:
import torchaudio
import numpy as np

HAPT_DIR = os.path.join(ROOT, "haptics")
os.makedirs(HAPT_DIR, exist_ok=True)

def gen_haptics_for_audio(wav_path, sentiment, T=200):
    wav, sr = torchaudio.load(wav_path)  # (ch, n)
    wav = wav.mean(dim=0)                # mono
    # energy envelope
    hop = max(1, wav.shape[0] // T)
    env = []
    for i in range(T):
        seg = wav[i*hop:(i+1)*hop]
        env.append(seg.abs().mean().item() if seg.numel() else 0.0)
    env = np.array(env, dtype=np.float32)
    env = (env - env.min()) / (env.max() - env.min() + 1e-8)

    # sentiment modulates "smoothness" and intensity
    vib = np.clip(0.2 + 0.8*(0.6*env + 0.4*sentiment), 0, 1)

    # pulse proxy: more pulses if negative/tense (1-sentiment)
    pulse = np.clip((1.0 - sentiment) * (0.3 + 0.7*env), 0, 1)

    # pressure proxy: stable “force” tied to env but biased by sentiment
    force = np.clip(0.1 + 0.9*(0.7*env + 0.3*sentiment), 0, 1)

    return np.stack([vib, pulse, force], axis=1).astype(np.float32)

hapt_paths = []
for j, wav_path, s in zip(items, aud_paths, s_scores):
    hp = os.path.join(HAPT_DIR, f"{j['i']:04d}.npy")
    np.save(hp, gen_haptics_for_audio(wav_path, float(s), T=200))
    hapt_paths.append(hp)

print("Saved haptics:", hapt_paths[0], np.load(hapt_paths[0]).shape)


RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.9.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call l

In [51]:
!apt-get update -qq && apt-get install -qq -y ffmpeg

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [52]:
!pip install torchcodec
import torchaudio
import numpy as np

HAPT_DIR = os.path.join(ROOT, "haptics")
os.makedirs(HAPT_DIR, exist_ok=True)

def gen_haptics_for_audio(wav_path, sentiment, T=200):
    wav, sr = torchaudio.load(wav_path)  # (ch, n)
    wav = wav.mean(dim=0)                # mono
    # energy envelope
    hop = max(1, wav.shape[0] // T)
    env = []
    for i in range(T):
        seg = wav[i*hop:(i+1)*hop]
        env.append(seg.abs().mean().item() if seg.numel() else 0.0)
    env = np.array(env, dtype=np.float32)
    env = (env - env.min()) / (env.max() - env.min() + 1e-8)

    # sentiment modulates "smoothness" and intensity
    vib = np.clip(0.2 + 0.8*(0.6*env + 0.4*sentiment), 0, 1)

    # pulse proxy: more pulses if negative/tense (1-sentiment)
    pulse = np.clip((1.0 - sentiment) * (0.3 + 0.7*env), 0, 1)

    # pressure proxy: stable “force” tied to env but biased by sentiment
    force = np.clip(0.1 + 0.9*(0.7*env + 0.3*sentiment), 0, 1)

    return np.stack([vib, pulse, force], axis=1).astype(np.float32)

hapt_paths = []
for j, wav_path, s in zip(items, aud_paths, s_scores):
    hp = os.path.join(HAPT_DIR, f"{j['i']:04d}.npy")
    np.save(hp, gen_haptics_for_audio(wav_path, float(s), T=200))
    hapt_paths.append(hp)

print("Saved haptics:", hapt_paths[0], np.load(hapt_paths[0]).shape)


RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.9.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call l

In [53]:
import torch.nn as nn
import torch.nn.functional as F

class HapticsEncoder(nn.Module):
    def __init__(self, in_ch=3, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.proj = nn.Linear(64, out_dim)

    def forward(self, x):  # x: (B,T,C)
        x = x.transpose(1,2)        # (B,C,T)
        h = self.net(x).squeeze(-1) # (B,64)
        z = self.proj(h)
        return F.normalize(z, dim=-1)

def load_haptics_batch(paths):
    arr = [np.load(p) for p in paths]  # each (T,3)
    x = torch.tensor(np.stack(arr, axis=0), dtype=torch.float32, device=device)
    return x


In [54]:
from brian2 import *
import numpy as np

def brian2_rate_features(x_np, sim_ms=30.0):
    """
    x_np: (B, D) float32
    returns: (B, D) spike rates
    """
    B, D = x_np.shape
    x = x_np
    x = (x - x.min()) / (x.max() - x.min() + 1e-8)

    defaultclock.dt = 0.1*ms
    tau = 10*ms
    eqs = '''
    dv/dt = (-v + I)/tau : 1
    I : 1
    '''

    rates_out = []
    for b in range(B):
        G = NeuronGroup(D, eqs, threshold='v>1', reset='v=0', method='euler')
        G.I = x[b]
        M = SpikeMonitor(G)
        run(sim_ms*ms)
        rate = np.array(M.count) / (sim_ms/1000.0)
        rates_out.append(rate.astype(np.float32))
        device.reinit(); device.activate()
    return np.stack(rates_out, axis=0)

# Example: make spiking features from text embeddings (first 128 dims for speed)


In [55]:
import cirq, qsimcirq
import numpy as np

def quantum_features(x_np, n_qubits=8, reps=2):
    B, D = x_np.shape
    qubits = cirq.LineQubit.range(n_qubits)
    sim = qsimcirq.QSimSimulator()

    feats=[]
    for b in range(B):
        angles = x_np[b, :n_qubits]
        c = cirq.Circuit()
        for i,q in enumerate(qubits):
            c.append(cirq.ry(float(angles[i]))(q))
            c.append(cirq.rz(float(angles[i]))(q))
        for _ in range(reps):
            for i in range(n_qubits-1):
                c.append(cirq.CNOT(qubits[i], qubits[i+1]))
        r = sim.simulate(c)
        state = r.final_state_vector
        probs = np.abs(state)**2

        q0=[]
        for qi in range(n_qubits):
            mask = 1 << (n_qubits-1-qi)
            p0 = probs[[i for i in range(len(probs)) if (i & mask)==0]].sum()
            q0.append(p0)
        q0 = np.array(q0, dtype=np.float32)
        feats.append(np.concatenate([q0, 1.0-q0], axis=0))
    return np.stack(feats, axis=0)


In [56]:
!pip install torchcodec
import torchaudio
import numpy as np

HAPT_DIR = os.path.join(ROOT, "haptics")
os.makedirs(HAPT_DIR, exist_ok=True)

def gen_haptics_for_audio(wav_path, sentiment, T=200):
    wav, sr = torchaudio.load(wav_path)  # (ch, n)
    wav = wav.mean(dim=0)                # mono
    # energy envelope
    hop = max(1, wav.shape[0] // T)
    env = []
    for i in range(T):
        seg = wav[i*hop:(i+1)*hop]
        env.append(seg.abs().mean().item() if seg.numel() else 0.0)
    env = np.array(env, dtype=np.float32)
    env = (env - env.min()) / (env.max() - env.min() + 1e-8)

    # sentiment modulates "smoothness" and intensity
    vib = np.clip(0.2 + 0.8*(0.6*env + 0.4*sentiment), 0, 1)

    # pulse proxy: more pulses if negative/tense (1-sentiment)
    pulse = np.clip((1.0 - sentiment) * (0.3 + 0.7*env), 0, 1)

    # pressure proxy: stable “force” tied to env but biased by sentiment
    force = np.clip(0.1 + 0.9*(0.7*env + 0.3*sentiment), 0, 1)

    return np.stack([vib, pulse, force], axis=1).astype(np.float32)

hapt_paths = []
for j, wav_path, s in zip(items, aud_paths, s_scores):
    hp = os.path.join(HAPT_DIR, f"{j['i']:04d}.npy")
    np.save(hp, gen_haptics_for_audio(wav_path, float(s), T=200))
    hapt_paths.append(hp)

print("Saved haptics:", hapt_paths[0], np.load(hapt_paths[0]).shape)


RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.9.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call l

In [ ]:
!ffmpeg -version

In [57]:
from brian2 import *
import numpy as np

def brian2_rate_features(x_np, sim_ms=30.0):
    """
    x_np: (B, D) float32
    returns: (B, D) spike rates
    """
    B, D = x_np.shape
    x = x_np
    x = (x - x.min()) / (x.max() - x.min() + 1e-8)

    defaultclock.dt = 0.1*ms
    tau = 10*ms
    eqs = '''
    dv/dt = (-v + I)/tau : 1
    I : 1
    '''

    rates_out = []
    for b in range(B):
        G = NeuronGroup(D, eqs, threshold='v>1', reset='v=0', method='euler')
        G.I = x[b]
        M = SpikeMonitor(G)
        run(sim_ms*ms)
        rate = np.array(M.count) / (sim_ms/1000.0)
        rates_out.append(rate.astype(np.float32))
        device.reinit(); device.activate()
    return np.stack(rates_out, axis=0)

# Example: make spiking features from text embeddings (first 128 dims for speed)


In [58]:
import cirq, qsimcirq
import numpy as np

def quantum_features(x_np, n_qubits=8, reps=2):
    B, D = x_np.shape
    qubits = cirq.LineQubit.range(n_qubits)
    sim = qsimcirq.QSimSimulator()

    feats=[]
    for b in range(B):
        angles = x_np[b, :n_qubits]
        c = cirq.Circuit()
        for i,q in enumerate(qubits):
            c.append(cirq.ry(float(angles[i]))(q))
            c.append(cirq.rz(float(angles[i]))(q))
        for _ in range(reps):
            for i in range(n_qubits-1):
                c.append(cirq.CNOT(qubits[i], qubits[i+1]))
        r = sim.simulate(c)
        state = r.final_state_vector
        probs = np.abs(state)**2

        q0=[]
        for qi in range(n_qubits):
            mask = 1 << (n_qubits-1-qi)
            p0 = probs[[i for i in range(len(probs)) if (i & mask)==0]].sum()
            q0.append(p0)
        q0 = np.array(q0, dtype=np.float32)
        feats.append(np.concatenate([q0, 1.0-q0], axis=0))
    return np.stack(feats, axis=0)


In [59]:
import torch

E_t = embed_text(texts)
E_i = embed_images(img_paths)
E_a = embed_audio(aud_paths)
E_v = embed_video(vid_paths)

print(E_t.shape, E_i.shape, E_a.shape, E_v.shape)


Attempting to cast a BatchEncoding to type <brian2.devices.device.CurrentDeviceProxy object at 0x7d47403960c0>. This is not supported.


TypeError: to() received an invalid combination of arguments - got (CurrentDeviceProxy), but expected one of:
 * (torch.device device = None, torch.dtype dtype = None, bool non_blocking = False, bool copy = False, *, torch.memory_format memory_format = None)
 * (torch.dtype dtype, bool non_blocking = False, bool copy = False, *, torch.memory_format memory_format = None)
 * (Tensor tensor, bool non_blocking = False, bool copy = False, *, torch.memory_format memory_format = None)


In [ ]:
import numpy as np

# --- Text synaptic+quantum augmentation (fast enough for N=49) ---
E_t_np = E_t.detach().cpu().numpy().astype(np.float32)

S_t = brian2_rate_features(E_t_np[:, :128], sim_ms=25.0)      # synaptic features
Q_t = quantum_features(E_t_np[:, :8], n_qubits=8, reps=2)     # quantum features

E_t_aug = torch.tensor(np.concatenate([E_t_np, S_t, Q_t], axis=1), device=device, dtype=torch.float32)
E_t_aug = torch.nn.functional.normalize(E_t_aug, dim=-1)

print("E_t_aug:", E_t_aug.shape)


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Projector(nn.Module):
    def __init__(self, in_dim, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.Linear(512, out_dim),
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

def info_nce(a, b, temp=0.07):
    logits = (a @ b.T) / temp
    labels = torch.arange(a.size(0), device=a.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

h_enc = HapticsEncoder(in_ch=3, out_dim=256).to(device)

p_t = Projector(E_t_aug.shape[1]).to(device)
p_i = Projector(E_i.shape[1]).to(device)
p_a = Projector(E_a.shape[1]).to(device)
p_v = Projector(E_v.shape[1]).to(device)

opt = torch.optim.AdamW(
    list(h_enc.parameters()) + list(p_t.parameters()) + list(p_i.parameters()) +
    list(p_a.parameters()) + list(p_v.parameters()),
    lr=2e-3, weight_decay=1e-4
)

H = load_haptics_batch(hapt_paths)      # (N,T,3)
N = E_i.shape[0]

for epoch in range(1, 401):
    idx = torch.randperm(N, device=device)

    zt = p_t(E_t_aug[idx])
    zi = p_i(E_i[idx])
    za = p_a(E_a[idx])
    zv = p_v(E_v[idx])
    zh = h_enc(H[idx])

    # all-pairs contrastive (5 modalities)
    loss = (
        info_nce(zt, zi) + info_nce(zt, za) + info_nce(zt, zv) + info_nce(zt, zh) +
        info_nce(zi, za) + info_nce(zi, zv) + info_nce(zi, zh) +
        info_nce(za, zv) + info_nce(za, zh) +
        info_nce(zv, zh)
    )

    opt.zero_grad()
    loss.backward()
    opt.step()

    if epoch % 50 == 0 or epoch == 1:
        print(f"epoch {epoch:4d}  loss {loss.item():.4f}")


In [ ]:
@torch.no_grad()
def retrieve_text_to_video(query_text, topk=5):
    q = embed_text([query_text])
    # augment query in same way:
    q_np = q.detach().cpu().numpy().astype(np.float32)
    S_q = brian2_rate_features(q_np[:, :128], sim_ms=25.0)
    Q_q = quantum_features(q_np[:, :8], n_qubits=8, reps=2)
    q_aug = torch.tensor(np.concatenate([q_np, S_q, Q_q], axis=1), device=device, dtype=torch.float32)
    q_aug = torch.nn.functional.normalize(q_aug, dim=-1)

    zq = p_t(q_aug)                # (1,256)
    zv_all = p_v(E_v)              # (N,256)
    sims = (zq @ zv_all.T).squeeze(0)
    vals, inds = torch.topk(sims, k=min(topk, len(sims)))
    return [(int(i.item()), float(v.item()), vid_paths[int(i.item())]) for v,i in zip(vals, inds)]

retrieve_text_to_video("drifting mood model cycle", topk=5)


In [ ]:
import cv2
import numpy as np
from PIL import Image

def parallax_views(img_path, n_views=9, max_shift=20):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h,w,_ = img.shape
    views=[]
    shifts = np.linspace(-max_shift, max_shift, n_views).astype(np.float32)

    for s in shifts:
        M = np.float32([[1,0,s],[0,1,0]])
        shifted = cv2.warpAffine(img, M, (w,h), borderMode=cv2.BORDER_REFLECT)
        views.append(shifted)
    grid = np.concatenate(views, axis=1)
    return Image.fromarray(grid)

# Show multi-view “holo strip” for a sample
parallax_views(img_paths[0], n_views=9, max_shift=18)
